# Boston Housing Data Analysis
### A Beginner-Friendly Data Analytics Portfolio Project

**Project type:** Exploratory Data Analysis (EDA) and statistical analysis  
**Tools:** Python, Pandas, NumPy, Matplotlib and Seaborn

## 1. Project Introduction

This project explores the Boston Housing dataset to understand how housing values vary with neighbourhood characteristics such as average rooms, crime rate, pollution, property tax and other recorded indicators.

The work began as 30 practical questions covering Pandas and NumPy. Those original exercises are retained and organised into a complete analytics workflow. Additional data-quality checks, visualisations and written interpretations make the analysis easier to understand and present.

**Scope:** This is a Data Analytics project. It does not train or evaluate a Machine Learning model.

> **Important context:** The Boston Housing dataset is historical and has known ethical limitations, including variables that encode historical racial and socioeconomic information. Its findings should be treated as an educational analysis of this dataset, not as current housing-market guidance or a basis for real-world decisions about people or neighbourhoods.

## 2. Project Objectives

- Load and understand a real-world tabular dataset.
- Inspect its dimensions, columns, data types and summary statistics.
- Check data quality, including missing values and duplicate rows.
- Use Pandas and NumPy for filtering, aggregation, sorting and statistical calculations.
- Explore house-value distributions and relationships using Matplotlib and Seaborn.
- Compare selected variables with median house value using correlation.
- Practise normalization, standardization and Z-score-based outlier detection.
- Create a derived feature and group houses by room-count ranges.
- Communicate findings in simple, evidence-based language.

## 3. Dataset Description

The dataset contains **506 observations and 14 original columns**. Each observation represents a Boston-area census tract/neighbourhood record, not an individual house sale.

The target-like measure used in the practical questions is `MEDV`, the median value of owner-occupied homes, recorded in thousands of US dollars in the historical dataset.

| Original column | Project column | Description |
|---|---|---|
| CRIM | `crime_rate` | Per-capita crime rate by town |
| ZN | `residential_land_percent` | Share of land zoned for large residential lots |
| INDUS | `business_land_percent` | Share of non-retail business acres |
| CHAS | `near_charles_river` | 1 if the tract bounds the Charles River; otherwise 0 |
| NOX | `pollution_level` | Nitric oxides concentration |
| RM | `average_rooms` | Average number of rooms per dwelling |
| AGE | `old_houses_percent` | Share of owner-occupied units built before 1940 |
| DIS | `employment_distance` | Weighted distance to five employment centres |
| RAD | `highway_access` | Index of accessibility to radial highways |
| TAX | `property_tax` | Property-tax rate per $10,000 of full value |
| PTRATIO | `student_teacher_ratio` | Pupil-teacher ratio by town |
| B | `demographic_index` | Historical transformed variable; see ethical note above |
| LSTAT | `lower_status_percent` | Historical socioeconomic indicator (percentage) |
| MEDV | `house_price` | Median owner-occupied home value, in $1,000s |

**Dataset source:** [BostonHousing.csv — selva86/datasets](https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv). The dataset is also described in the University of Washington's [dataset notes](https://faculty.washington.edu/otoomet/machinelearning-py/datasets.html).

## 4. Import Libraries and Load the Dataset

Place `BostonHousing.csv` in the same folder as this notebook. If it is not present, the cell below attempts to download the public CSV source and save a local copy. An internet connection is required only for this first download.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (8, 5)

DATA_FILE = "BostonHousing.csv"
DATA_URL = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"

if not os.path.exists(DATA_FILE):
    print("Local CSV not found. Downloading the dataset...")
    df_raw = pd.read_csv(DATA_URL)
    df_raw.to_csv(DATA_FILE, index=False)
else:
    df_raw = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Original shape:", df_raw.shape)
display(df_raw.head())

## 5. Rename Columns

The following names make the columns easier to read in the remaining analysis. The renaming follows the practical notebook. The historical meaning of the `B` and `LSTAT` variables should not be obscured by friendlier labels.

In [ ]:
df = df_raw.rename(columns={
    "crim": "crime_rate",
    "zn": "residential_land_percent",
    "indus": "business_land_percent",
    "chas": "near_charles_river",
    "nox": "pollution_level",
    "rm": "average_rooms",
    "age": "old_houses_percent",
    "dis": "employment_distance",
    "rad": "highway_access",
    "tax": "property_tax",
    "ptratio": "student_teacher_ratio",
    "b": "demographic_index",
    "lstat": "lower_status_percent",
    "medv": "house_price"
})

display(df.head())

## 6. Data Loading and Understanding

### Original practical questions Q1–Q5

These questions check that the CSV is loaded, the dimensions are understood, and the required columns can be inspected and extracted.

### Q1. Load the dataset

In [ ]:
# The dataset was loaded in Section 4.
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

### Q2. Print the shape of the dataset

In [ ]:
print(df.shape)

### Q3. Print the first 5 rows

In [ ]:
display(df.head())

### Q4. Print the last 5 rows

In [ ]:
display(df.tail(5))

### Q5. Extract the house-price column (MEDV)

In [ ]:
price = df["house_price"]
display(price.head())

### Additional dataset inspection

These checks extend the original practicals and help an analyst understand the structure before drawing conclusions.

In [ ]:
print("Dataset information:")
df.info()

print("\
Descriptive statistics:")
display(df.describe().T)

print("\
Data types:")
display(df.dtypes)

print("\
Missing values by column:")
display(df.isnull().sum())

print("\
Duplicate rows:", df.duplicated().sum())

## 7. Data Cleaning and Preprocessing

First inspect the quality checks above. The Boston Housing version used for this project is commonly distributed with 506 rows, 14 numeric columns and no missing values. The code below handles missing values only if they are actually present, and removes duplicate rows only if duplicates are found.

No values are capped or deleted from the main analytical dataset in this section. Later, Q28 creates a separate capped-price column to preserve the original values.

In [ ]:
missing_before = int(df.isnull().sum().sum())
duplicates_before = int(df.duplicated().sum())

print("Missing values before cleaning:", missing_before)
print("Duplicate rows before cleaning:", duplicates_before)

if duplicates_before > 0:
    df = df.drop_duplicates().copy()

# No blanket imputation is applied. If missing values exist, inspect
# their columns and decide on a domain-appropriate treatment first.
price = df["house_price"]  # Refresh after any duplicate removal.
print("Shape after duplicate check:", df.shape)
print("Missing values after duplicate check:", int(df.isnull().sum().sum()))

## 8. Statistical Analysis

### Original practical questions Q6–Q10

The price values in `house_price` are measured in thousands of historical US dollars. The standard deviation below uses NumPy's population definition (`ddof=0`), matching the original NumPy calculation.

### Q6. Find the mean house price

In [ ]:
mean_price = np.mean(price)
print("Mean house price (NumPy):", mean_price)
print("Mean house price (Pandas):", df["house_price"].mean())

### Q7. Find the maximum house price

In [ ]:
max_price = np.max(price)
print("Maximum house price (NumPy):", max_price)
print("Maximum house price (Pandas):", df["house_price"].max())

### Q8. Find the minimum house price

In [ ]:
min_price = np.min(price)
print("Minimum house price (NumPy):", min_price)
print("Minimum house price (Pandas):", df["house_price"].min())

### Q9. Find the standard deviation of house price

In [ ]:
std_price = np.std(price, ddof=0)
print("Population standard deviation:", std_price)

# Pandas sample standard deviation uses ddof=1 by default.
print("Pandas population standard deviation:", df["house_price"].std(ddof=0))

### Q10. Count the total number of rows

In [ ]:
total_rows = len(df)
print("Total rows:", total_rows)

### Additional statistical summary

Median and quartiles help describe the centre and spread of prices. They are useful alongside the mean because a distribution can be affected by unusually high or low values.

In [ ]:
price_summary = df["house_price"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
display(price_summary)

print("25th percentile:", df["house_price"].quantile(0.25))
print("75th percentile:", df["house_price"].quantile(0.75))
print("Price skewness:", df["house_price"].skew())

## 9. Exploratory Data Analysis (EDA)

EDA uses summaries and charts to understand distributions, unusual values and possible relationships before making conclusions.

### 9.1 Distribution of house prices

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="house_price", bins=30, kde=True)
plt.title("Distribution of Median House Values")
plt.xlabel("Median house value ($1,000s)")
plt.ylabel("Number of neighbourhood records")
plt.tight_layout()
plt.show()

### 9.2 House price box plot

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(data=df, x="house_price")
plt.title("Spread of Median House Values")
plt.xlabel("Median house value ($1,000s)")
plt.tight_layout()
plt.show()

### 9.3 Average rooms versus house price

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="average_rooms", y="house_price", alpha=0.7)
plt.title("Average Rooms vs. Median House Value")
plt.xlabel("Average rooms per dwelling")
plt.ylabel("Median house value ($1,000s)")
plt.tight_layout()
plt.show()

### 9.4 Crime rate versus house price

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="crime_rate", y="house_price", alpha=0.65)
plt.title("Crime Rate vs. Median House Value")
plt.xlabel("Per-capita crime rate")
plt.ylabel("Median house value ($1,000s)")
plt.tight_layout()
plt.show()

### 9.5 House price by Charles River indicator

`near_charles_river` is a binary indicator: 0 means the tract does not bound the river and 1 means it does. The chart compares the distributions; it does not establish that river proximity causes a price difference.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="near_charles_river", y="house_price")
plt.title("House Values by Charles River Indicator")
plt.xlabel("Bounds the Charles River (0 = No, 1 = Yes)")
plt.ylabel("Median house value ($1,000s)")
plt.tight_layout()
plt.show()

## 10. Intermediate Analysis — Filtering, Conditions and Sorting

### Original practical questions Q11–Q20

These exercises are retained from the original notebook. Filtering conditions select records; they do not modify the source data.

### Q11. Extract the average-rooms column

In [ ]:
rooms = df["average_rooms"]
display(rooms.head())

### Q12. Find the average number of rooms

In [ ]:
avg_rooms = np.mean(rooms)
print("Average rooms (NumPy):", avg_rooms)
print("Average rooms (Pandas):", df["average_rooms"].mean())

### Q13. Find houses where price is greater than 25

In [ ]:
expensive = df[df["house_price"] > 25]
print("Number of records with price > 25:", len(expensive))
display(expensive.head())

### Q14. Count houses where price is less than 20

In [ ]:
count_below_20 = (df["house_price"] < 20).sum()
print("Number of records with price < 20:", count_below_20)

### Q15. Find houses where average rooms are greater than 6

In [ ]:
rooms_above_6 = df[df["average_rooms"] > 6]
print("Number of records with rooms > 6:", len(rooms_above_6))
display(rooms_above_6.head())

### Q16. Apply two conditions: rooms greater than 6 AND price greater than 30

In [ ]:
result = df[
    (df["average_rooms"] > 6) &
    (df["house_price"] > 30)
]
display(result.head())
print("Matching records:", len(result))

### Q17. Find average price for houses with more than 6 rooms

In [ ]:
filtered_data = df[df["average_rooms"] > 6]
avg_price = filtered_data["house_price"].mean()
print("Average price when rooms > 6:", avg_price)

### Q18. Sort the dataset by house price

In [ ]:
sorted_ascending = df.sort_values(by="house_price", ascending=True)
sorted_descending = df.sort_values(by="house_price", ascending=False)

display(sorted_ascending.head())
display(sorted_descending.head())

### Q19. Get the top 10 most expensive records

In [ ]:
top_10 = df.nlargest(10, "house_price")
display(top_10)

# Alternative original approach: sort from highest to lowest, then take 10 rows.
top_10_by_sort = df.sort_values(by="house_price", ascending=False).head(10)
display(top_10_by_sort)

### Q20. Get the bottom 10 cheapest records

In [ ]:
bottom_10 = df.nsmallest(10, "house_price")
display(bottom_10)

# Alternative original approach: sort from lowest to highest, then take 10 rows.
bottom_10_by_sort = df.sort_values(by="house_price", ascending=True).head(10)
display(bottom_10_by_sort)

## 11. Data Visualization Summary

This section adds a few compact comparisons that support the practical questions and make patterns easier to communicate.

### Average house value for room-count groups

In [ ]:
room_bins_for_plot = [0, 5, 6, 7, 8, np.inf]
room_labels_for_plot = ["Below 5", "5 to <6", "6 to <7", "7 to <8", "8 and above"]

plot_df = df.copy()
plot_df["room_range_plot"] = pd.cut(
    plot_df["average_rooms"],
    bins=room_bins_for_plot,
    labels=room_labels_for_plot,
    right=False
)

avg_price_plot = plot_df.groupby(
    "room_range_plot", observed=False
)["house_price"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(x=avg_price_plot.index, y=avg_price_plot.values)
plt.title("Average Median House Value by Average-Room Range")
plt.xlabel("Average rooms range")
plt.ylabel("Average median house value ($1,000s)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Correlation Analysis

Correlation measures the strength and direction of a linear relationship. It does not prove cause and effect. Pearson correlation ranges from -1 to +1:

- Close to +1: strong positive linear association.
- Close to -1: strong negative linear association.
- Close to 0: weak linear association.

### Original practical question Q23. Correlation between rooms and price

In [ ]:
corr_rooms_price = df["average_rooms"].corr(df["house_price"])
print("Correlation (average_rooms, house_price):", corr_rooms_price)

**Interpretation:** The original notebook output is approximately **+0.695**. This indicates a moderately strong positive linear association in this dataset: neighbourhood records with more average rooms tend to have higher median house values. This is an association, not proof that adding rooms alone causes a higher value.

### Original practical question Q24. Correlation between crime rate and price

In [ ]:
corr_crime_price = df["crime_rate"].corr(df["house_price"])
print("Correlation (crime_rate, house_price):", corr_crime_price)

**Interpretation:** The original notebook output is approximately **-0.388**. This indicates a negative linear association: higher crime-rate values tend to occur alongside lower median house values in this dataset. Other variables and historical context may also be involved.

### Correlation matrix and heatmap

In [ ]:
correlation_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(13, 9))
sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.4
)
plt.title("Correlation Heatmap of Boston Housing Variables")
plt.tight_layout()
plt.show()

# Sort correlations with house_price, excluding its correlation with itself.
price_correlations = correlation_matrix["house_price"].drop("house_price")
display(price_correlations.sort_values(ascending=False).to_frame("correlation_with_house_price"))

## 13. Advanced Analysis — Scaling, Feature Preparation and Outliers

### Original practical question Q21. Normalize numerical columns using Min-Max scaling

Formula: `(value - column minimum) / (column maximum - column minimum)`. Each non-constant column is rescaled to a range from 0 to 1.

This creates a separate transformed DataFrame; the original `df` remains unchanged. This exercise includes `house_price` because the original practical selected all numeric columns. It is a learning exercise, not a model-preparation pipeline.

In [ ]:
numeric_df = df.select_dtypes(include="number")

column_ranges = numeric_df.max() - numeric_df.min()
normalized_data = (numeric_df - numeric_df.min()) / column_ranges.replace(0, np.nan)

display(normalized_data.head())
print("Minimum values after scaling:")
display(normalized_data.min())
print("Maximum values after scaling:")
display(normalized_data.max())

### Original practical question Q22. Standardize numerical columns using Z-score

Formula: `(value - column mean) / column standard deviation`. The resulting columns have mean approximately 0 and population standard deviation approximately 1, provided the column has non-zero variation.

In [ ]:
standardized_data = (
    numeric_df - numeric_df.mean()
) / numeric_df.std(ddof=0).replace(0, np.nan)

display(standardized_data.head())
print("Column means after standardization:")
display(standardized_data.mean().round(2))
print("Column standard deviations after standardization:")
display(standardized_data.std(ddof=0).round(2))

### Original practical question Q25. Split the dataset into features (X) and target (y)

This is a data-organisation exercise only. No model is trained. The 13 original predictor columns are placed in `X`; `house_price` is placed in `y`.

In [ ]:
X = df.drop("house_price", axis=1)
y = df["house_price"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

### Original practical question Q26. Create a new feature: price per room

The calculation divides each neighbourhood's median house value by its average room count. It is a simple derived ratio for practice, not an official property valuation metric.

In [ ]:
df["price_per_room"] = df["house_price"] / df["average_rooms"]
display(df[["house_price", "average_rooms", "price_per_room"]].head())

### Original practical question Q27. Detect price outliers using Z-score

A common screening rule flags values with an absolute Z-score greater than 3. This rule is a heuristic. It may miss unusual values in skewed distributions and does not mean that a flagged observation is an error.

In [ ]:
price_z_score = (
    df["house_price"] - df["house_price"].mean()
) / df["house_price"].std(ddof=0)

price_outliers = df[price_z_score.abs() > 3]
print("Potential price outliers:", len(price_outliers))
display(price_outliers)

### Q27 (additional). Count potential outliers in every numeric column

This reproduces the original notebook's all-numeric-column outlier count. The count is based on the same absolute Z-score greater-than-3 rule.

In [ ]:
numeric_with_feature = df.select_dtypes(include="number")
all_z_scores = (
    numeric_with_feature - numeric_with_feature.mean()
) / numeric_with_feature.std(ddof=0).replace(0, np.nan)

outlier_counts = (all_z_scores.abs() > 3).sum()
display(outlier_counts.to_frame("potential_outlier_count"))

### Original practical question Q28. Cap house prices above 50 at 50

The original `house_price` column is preserved. A new column, `house_price_capped`, stores the capped values. The threshold is part of the practical exercise; it should not be treated as an automatically correct cleaning decision.

In [ ]:
df["house_price_capped"] = np.where(
    df["house_price"] > 50,
    50,
    df["house_price"]
)

display(df[["house_price", "house_price_capped"]].head())
print("Values above 50 before capping:", (df["house_price"] > 50).sum())

### Original practical question Q29. Create a Boolean mask for high-price records

The notebook defines a high-price record as one with `house_price > 30`. The mask contains `True` or `False` for each row and can be used to filter the DataFrame.

In [ ]:
high_price_mask = df["house_price"] > 30
high_price_records = df[high_price_mask]

print("High-price records:", int(high_price_mask.sum()))
display(high_price_records.head())

### Original practical question Q30. Group houses by room ranges and calculate average price

The original notebook uses these intervals: below 5, 5 to below 6, 6 to below 7, 7 to below 8, and 8 or more. `right=False` makes the left boundary inclusive and the right boundary exclusive.

In [ ]:
df["room_range"] = pd.cut(
    df["average_rooms"],
    bins=[0, 5, 6, 7, 8, np.inf],
    labels=["Below 5", "5 to 6", "6 to 7", "7 to 8", "8 and above"],
    right=False
)

avg_price_by_rooms = df.groupby(
    "room_range", observed=False
)["house_price"].mean()

display(avg_price_by_rooms.to_frame("average_house_price"))

## 14. Important Findings and Business-Style Insights

The following findings are calculated from the dataset when the notebook is run. Keep the language descriptive: these are patterns in a historical dataset, not causal conclusions or current market recommendations.

In [ ]:
print(f"Dataset records: {len(df)}")
print(f"Average median house value: {df['house_price'].mean():.2f} thousand USD")
print(f"Median median house value: {df['house_price'].median():.2f} thousand USD")
print(f"Average rooms per dwelling: {df['average_rooms'].mean():.2f}")
print(f"Rooms-price correlation: {df['average_rooms'].corr(df['house_price']):.3f}")
print(f"Crime-price correlation: {df['crime_rate'].corr(df['house_price']):.3f}")
print(f"Records with price > 30: {(df['house_price'] > 30).sum()}")
print(f"Records with price < 20: {(df['house_price'] < 20).sum()}")

print("\
Average price by room range:")
display(avg_price_by_rooms.to_frame("average_house_price"))

## 15. Final Conclusion

This project demonstrates a complete beginner-level Data Analytics workflow using the Boston Housing dataset. It retains the original 30 practical exercises and extends them with data-quality checks, descriptive statistics, visual exploration, a correlation heatmap and clearer interpretations.

The analysis explores how median house values vary with recorded neighbourhood characteristics. In the original notebook, average rooms had a positive correlation of approximately 0.695 with house value, while crime rate had a negative correlation of approximately -0.388. These values describe linear associations in this historical dataset and do not establish causation.

The project also practises Min-Max scaling, Z-score standardization, outlier screening, Boolean filtering, value capping and room-range grouping. These operations are kept separate from the original values wherever appropriate so that transformations remain transparent.

**Limitations:** The data is historical, represents neighbourhood-level records, and includes variables with ethical concerns. The dataset should not be used to make current property decisions or judgements about residents. No Machine Learning model is included.

## 16. Reproducibility Checklist

1. Keep `Boston_Housing_Analysis.ipynb` and `BostonHousing.csv` in the same folder.
2. Install the packages listed in `requirements.txt`.
3. Open the notebook in Jupyter Notebook or JupyterLab.
4. Select **Kernel → Restart & Run All** (or the equivalent option).
5. Review the printed summaries, tables, plots and interpretations.

If the CSV is not in the folder, the loading cell attempts to download it from the source URL. If that download is blocked, download the CSV manually from the dataset source linked in Section 3 and save it as `BostonHousing.csv`.